# FLAN-T5 Fine-tuning on MedMCQA
This notebook fine-tunes `google/flan-t5-base` on the **MedMCQA** dataset and compares base vs fine-tuned model performance.

## Cell 1 — Install Dependencies

In [ ]:
!pip install transformers datasets accelerate evaluate peft bert-score
!pip install -U fsspec==2023.9.2

## Cell 2 — Load MedMCQA Dataset (train[:20%])

In [ ]:
from datasets import load_dataset

medmcqa = load_dataset("medmcqa", split="train[:20%]")

medmcqa = medmcqa.filter(
    lambda x: x["question"] and x["opa"] and x["opb"] and x["opc"] and x["opd"] and x["cop"] is not None
)

print(f"Training samples: {len(medmcqa)}")

## Cell 3 — Preprocess MedMCQA

In [ ]:
def format_mcq(example):
    input_text = (
        f"question: {example['question']} "
        f"options: A. {example['opa']} B. {example['opb']} C. {example['opc']} D. {example['opd']}"
    )
    options = [example["opa"], example["opb"], example["opc"], example["opd"]]
    try:
        target_text = options[int(example["cop"])]
    except (ValueError, IndexError):
        target_text = "Unknown"
    return {"input_text": input_text, "target_text": target_text}

medmcqa = medmcqa.map(format_mcq)

## Cell 4 — Tokenize MedMCQA

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def tokenize(example):
    model_input = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    labels = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    model_input["labels"] = labels["input_ids"]
    return model_input

tokenized_medmcqa = medmcqa.map(tokenize, remove_columns=medmcqa.column_names)
print(f"Tokenized samples: {len(tokenized_medmcqa)}")

## Cell 5 — Define Training Function (num_train_epochs=3)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def train_on_dataset(tokenized_dataset, model_save_path, model, tokenizer):
    training_args = Seq2SeqTrainingArguments(
        output_dir=model_save_path,
        per_device_train_batch_size=4,
        learning_rate=3e-4,
        num_train_epochs=3,
        weight_decay=0.01,
        predict_with_generate=True,
        logging_dir="./logs",
        logging_steps=10,
        save_total_limit=1,
        save_strategy="epoch"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer
    )

    trainer.train()
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

## Cell 6 — Fine-tune FLAN-T5 on MedMCQA

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer

model_medmcqa = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
tokenizer_medmcqa = AutoTokenizer.from_pretrained("google/flan-t5-base")

train_on_dataset(
    tokenized_dataset=tokenized_medmcqa,
    model_save_path="./flan-t5-medmcqa1",
    model=model_medmcqa,
    tokenizer=tokenizer_medmcqa
)

## Cell 7 — Load Models for Evaluation

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_base = AutoTokenizer.from_pretrained("google/flan-t5-base")
base_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)

tokenizer_finetuned = AutoTokenizer.from_pretrained("./flan-t5-medmcqa1")
finetuned_model = T5ForConditionalGeneration.from_pretrained("./flan-t5-medmcqa1").to(device)

print(f"Using device: {device}")

## Cell 8 — Build Evaluation Dataset (MedMCQA validation)

In [ ]:
from datasets import load_dataset

eval_raw = load_dataset("medmcqa", split="validation[:200]")

label_map = {0: "A", 1: "B", 2: "C", 3: "D"}
option_index = {0: "opa", 1: "opb", 2: "opc", 3: "opd"}

eval_data = []
for row in eval_raw:
    options = [row["opa"], row["opb"], row["opc"], row["opd"]]
    cop = int(row["cop"])
    eval_data.append({
        "question": row["question"],
        "options": options,
        "answer": options[cop],
        "answer_label": label_map[cop],
        "subject_name": row.get("subject_name", "Unknown"),
        "topic_name": row.get("topic_name", "Unknown"),
    })

print(f"Evaluation samples: {len(eval_data)}")
print("Sample:", eval_data[0])

## Cell 9 — Inference Function

In [ ]:
def generate_mcq_answer(model, tokenizer, question, options, max_length=64):
    device = next(model.parameters()).device
    input_text = (
        f"question: {question} options: "
        + " ".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)])
    )
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    # Try to match to one of the given options
    answer_lower = answer.lower()
    for opt in options:
        if opt.lower() in answer_lower or answer_lower in opt.lower():
            return opt
    return answer

## Cell 10 — Run Inference for Both Models

In [ ]:
from tqdm import tqdm

base_preds = []
finetuned_preds = []
ground_truths = []

for sample in tqdm(eval_data, desc="Evaluating"):
    gt = sample["answer"]
    ground_truths.append(gt)

    base_pred = generate_mcq_answer(base_model, tokenizer_base, sample["question"], sample["options"])
    base_preds.append(base_pred)

    ft_pred = generate_mcq_answer(finetuned_model, tokenizer_finetuned, sample["question"], sample["options"])
    finetuned_preds.append(ft_pred)

print(f"Inference complete for {len(ground_truths)} samples.")

## Cell 11 — Compute Metrics: Accuracy, F1, Exact Match, BERTScore

In [ ]:
import evaluate
import numpy as np
import re

bertscore_metric = evaluate.load("bertscore")

def normalize_text(s):
    s = s.lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[^\w\s]', '', s)
    return s.strip()

def compute_token_f1(pred, ref):
    pred_tokens = normalize_text(pred).split()
    ref_tokens = normalize_text(ref).split()
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens) if pred_tokens else 0.0
    recall = len(common) / len(ref_tokens) if ref_tokens else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def compute_metrics(preds, refs):
    norm_preds = [normalize_text(p) for p in preds]
    norm_refs = [normalize_text(r) for r in refs]
    accuracy = np.mean([p == r for p, r in zip(norm_preds, norm_refs)])
    em = np.mean([1 if p == r else 0 for p, r in zip(norm_preds, norm_refs)])
    f1 = np.mean([compute_token_f1(p, r) for p, r in zip(preds, refs)])
    bs = bertscore_metric.compute(predictions=preds, references=refs, lang="en")
    bs_f1 = np.mean(bs["f1"])
    return {"Accuracy": accuracy, "F1": f1, "Exact Match": em, "BERTScore F1": bs_f1}

base_metrics = compute_metrics(base_preds, ground_truths)
ft_metrics = compute_metrics(finetuned_preds, ground_truths)

print("Base FLAN-T5 Metrics:")
for k, v in base_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nFine-tuned FLAN-T5 (MedMCQA) Metrics:")
for k, v in ft_metrics.items():
    print(f"  {k}: {v:.4f}")

## Cell 12 — Model Comparison Table

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Metric": list(base_metrics.keys()),
    "Base FLAN-T5": [round(v, 4) for v in base_metrics.values()],
    "Fine-tuned FLAN-T5": [round(v, 4) for v in ft_metrics.values()],
})

print(comparison_df.to_string(index=False))

## Cell 13 — Error Analysis: Subject-level Performance

In [ ]:
from collections import defaultdict

subject_correct_ft = defaultdict(int)
subject_total = defaultdict(int)

for i, sample in enumerate(eval_data):
    subj = sample["subject_name"]
    subject_total[subj] += 1
    if normalize_text(finetuned_preds[i]) == normalize_text(sample["answer"]):
        subject_correct_ft[subj] += 1

subject_acc = {
    subj: subject_correct_ft[subj] / subject_total[subj]
    for subj in subject_total
}

subject_df = pd.DataFrame([
    {"Subject": s, "Accuracy": round(acc, 4), "Total": subject_total[s]}
    for s, acc in sorted(subject_acc.items(), key=lambda x: -x[1])
])

print("Fine-tuned model accuracy by subject:")
print(subject_df.to_string(index=False))

## Cell 14 — Error Analysis: Topic-level Performance

In [ ]:
topic_correct_ft = defaultdict(int)
topic_total = defaultdict(int)

for i, sample in enumerate(eval_data):
    topic = sample["topic_name"]
    topic_total[topic] += 1
    if normalize_text(finetuned_preds[i]) == normalize_text(sample["answer"]):
        topic_correct_ft[topic] += 1

topic_acc = {
    t: topic_correct_ft[t] / topic_total[t]
    for t in topic_total
}

# Show bottom 10 topics (lowest performance)
topic_df = pd.DataFrame([
    {"Topic": t, "Accuracy": round(acc, 4), "Total": topic_total[t]}
    for t, acc in sorted(topic_acc.items(), key=lambda x: x[1])
    if topic_total[t] >= 2
])

print("Topics with lowest fine-tuned model accuracy (min 2 samples):")
print(topic_df.head(10).to_string(index=False))

## Cell 15 — Error Analysis: Wrong Prediction Distribution (A/B/C/D)

In [ ]:
from collections import Counter

wrong_pred_labels = []
for i, sample in enumerate(eval_data):
    if normalize_text(finetuned_preds[i]) != normalize_text(sample["answer"]):
        pred = finetuned_preds[i].lower()
        matched_label = "Other"
        for j, opt in enumerate(sample["options"]):
            if opt.lower() in pred or pred in opt.lower():
                matched_label = chr(65 + j)
                break
        wrong_pred_labels.append(matched_label)

wrong_dist = Counter(wrong_pred_labels)
print("Distribution of wrong predictions by option (fine-tuned model):")
for label in sorted(wrong_dist.keys()):
    print(f"  {label}: {wrong_dist[label]}")

## Cell 16 — Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

def get_option_label(pred_text, options):
    pred_lower = normalize_text(pred_text)
    for j, opt in enumerate(options):
        if normalize_text(opt) in pred_lower or pred_lower in normalize_text(opt):
            return chr(65 + j)
    return "Other"

true_labels = [s["answer_label"] for s in eval_data]
ft_pred_labels = [get_option_label(finetuned_preds[i], eval_data[i]["options"]) for i in range(len(eval_data))]
base_pred_labels = [get_option_label(base_preds[i], eval_data[i]["options"]) for i in range(len(eval_data))]

all_labels = ["A", "B", "C", "D"]
cm_ft = confusion_matrix(true_labels, ft_pred_labels, labels=all_labels)
cm_base = confusion_matrix(true_labels, base_pred_labels, labels=all_labels)

print("Confusion Matrix — Fine-tuned FLAN-T5:")
print(pd.DataFrame(cm_ft, index=all_labels, columns=all_labels))

print("\nConfusion Matrix — Base FLAN-T5:")
print(pd.DataFrame(cm_base, index=all_labels, columns=all_labels))

## Cell 17 — Visualization: Accuracy Comparison Bar Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics_names = list(base_metrics.keys())
base_vals = [base_metrics[m] for m in metrics_names]
ft_vals = [ft_metrics[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, base_vals, width, label="Base FLAN-T5", color="steelblue")
bars2 = ax.bar(x + width/2, ft_vals, width, label="Fine-tuned FLAN-T5", color="darkorange")

ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_title("Base vs Fine-tuned FLAN-T5 on MedMCQA")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.0)
ax.legend()

for bar in bars1:
    ax.annotate(f"{bar.get_height():.3f}", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
for bar in bars2:
    ax.annotate(f"{bar.get_height():.3f}", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

## Cell 18 — Visualization: Subject-level Performance

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

subjects = subject_df["Subject"].tolist()
accs = subject_df["Accuracy"].tolist()

colors = ["green" if a >= 0.5 else "salmon" for a in accs]
ax.barh(subjects, accs, color=colors)
ax.set_xlabel("Accuracy")
ax.set_title("Fine-tuned FLAN-T5 Accuracy by Subject (MedMCQA)")
ax.set_xlim(0, 1.0)
ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.savefig("subject_performance.png", dpi=150)
plt.show()

## Cell 19 — Visualization: Confusion Matrix Heatmap

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_base, annot=True, fmt="d", xticklabels=all_labels, yticklabels=all_labels,
            cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion Matrix — Base FLAN-T5")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(cm_ft, annot=True, fmt="d", xticklabels=all_labels, yticklabels=all_labels,
            cmap="Oranges", ax=axes[1])
axes[1].set_title("Confusion Matrix — Fine-tuned FLAN-T5")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

## Cell 20 — Sample Predictions

In [ ]:
print("Sample MedMCQA Predictions (first 7):")
for i in range(min(7, len(eval_data))):
    sample = eval_data[i]
    print(f"Q: {sample['question']}")
    print(f"Options: {sample['options']}")
    print(f"GT: {sample['answer']}")
    print(f"Base Pred: {base_preds[i]}")
    print(f"Fine-tuned Pred: {finetuned_preds[i]}")
    print("-" * 60)